# RSI 14 Slope Momentum on SPY
## Strategy Brief
This strategy uses the Relative Strength Index (RSI) with a 14-day period to identify momentum in the SPY ETF. The slope of the RSI is calculated to determine the momentum direction. A positive slope suggests bullish momentum, while a negative slope indicates bearish momentum. Trades are initiated based on the direction of the RSI slope. Historical backtesting shows potential for capturing trends, but performance may vary based on market conditions.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
We will define the parameters for our RSI 14 Slope Momentum strategy. These parameters include the RSI period and the lookback period for calculating the slope.

In [ ]:
RSI_PERIOD = 14
SLOPE_LOOKBACK = 3
START_DATE = '2010-01-01'
TICKER = 'SPY'

### PHASE 2 - Data Exploration
We will download historical data for SPY using yfinance, compute the RSI, and calculate its slope. The RSI will be plotted overlaid on the price to visualize its behavior.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE)

# Compute RSI
def compute_rsi(data, period=14):
    delta = data['Adj Close'].diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

data['RSI'] = compute_rsi(data, RSI_PERIOD)

# Compute RSI slope
data['RSI_Slope'] = data['RSI'].diff(SLOPE_LOOKBACK)

# Plotting
plt.figure(figsize=(14, 7))
plt.subplot(2, 1, 1)
plt.plot(data['Adj Close'], label='SPY Price')
plt.title('SPY Price and RSI')
plt.legend()
plt.subplot(2, 1, 2)
plt.plot(data['RSI'], label='RSI')
plt.axhline(70, color='r', linestyle='--')
plt.axhline(30, color='g', linestyle='--')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
We will create a signal based on the RSI slope. A positive slope indicates a buy signal, and a negative slope indicates a sell signal. We will then define the entry and exit logic and calculate the positions.

In [ ]:
# Generate signals based on RSI slope
data['Signal'] = 0
data.loc[data['RSI_Slope'] > 0, 'Signal'] = 1
data.loc[data['RSI_Slope'] < 0, 'Signal'] = -1

# Calculate positions
data['Position'] = data['Signal'].shift(1)

### PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns based on the positions and plot the resulting equity curve.

In [ ]:
# Calculate daily returns
data['Market_Return'] = data['Adj Close'].pct_change()
data['Strategy_Return'] = data['Market_Return'] * data['Position']

data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()
data['Market_Equity_Curve'] = (1 + data['Market_Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.plot(data['Market_Equity_Curve'], label='Market Equity Curve', linestyle='--')
plt.title('Equity Curve')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and Maximum Drawdown. We will also compare these metrics against a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(data):
    cagr = (data['Equity_Curve'].iloc[-1] ** (252.0/len(data)) - 1) * 100
    sharpe_ratio = (data['Strategy_Return'].mean() / data['Strategy_Return'].std()) * np.sqrt(252)
    downside_std = data.loc[data['Strategy_Return'] < 0, 'Strategy_Return'].std()
    sortino_ratio = (data['Strategy_Return'].mean() / downside_std) * np.sqrt(252)
    max_drawdown = (data['Equity_Curve'].cummax() - data['Equity_Curve']).max()
    calmar_ratio = cagr / max_drawdown
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(data)
market_metrics = calculate_performance_metrics(data.assign(Equity_Curve=data['Market_Equity_Curve']))

performance_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Market': market_metrics
})
print(performance_df)

### PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of SPY data, compute today's RSI slope, and print the current position based on the strategy.

In [ ]:
def get_latest_signal(ticker='SPY', period=60):
    data = yf.download(ticker, period=f'{period}d')
    data['RSI'] = compute_rsi(data, RSI_PERIOD)
    data['RSI_Slope'] = data['RSI'].diff(SLOPE_LOOKBACK)
    latest_slope = data['RSI_Slope'].iloc[-1]
    if latest_slope > 0:
        print('Current Position: Long')
    elif latest_slope < 0:
        print('Current Position: Short')
    else:
        print('Current Position: Neutral')

get_latest_signal()